# **Librerias y Funciones**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import bisect

from ripser import ripser
from persim import plot_diagrams
from persim import PersistenceImager

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from scipy.integrate import odeint

from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# **Comparación de PCA's de Atractores de Lorentz mediante Concatenación**

In [2]:
n=10
estados_iniciales = []
for j in range(n):
    estados_iniciales.append([random.uniform(-10, 10), random.uniform(-10, 10), random.uniform(-10, 10)])

In [3]:
valores_rho_2=list(range(0,40,1))

rho_1 = 40
sigma_1 = 10.0
beta_1 = 8.0 / 3.0
t = np.arange(0.0, 40.0, 0.1)

def lorentz(state, t):
    x, y, z = state  
    return sigma_1 * (y - x), x * (rho_1 - z) - y, x * y - beta_1 * z 

Lorentz_1=[]
for j in range(n):
  states = odeint(lorentz, estados_iniciales[j], t)

  Lorentz_1.append(states)

In [ ]:
sigma_2 = 10.0
beta_2 = 8.0 / 3.0
t = np.arange(0.0, 40.0, 0.1)

for k in range(len(valores_rho_2)):
    rho_2 = valores_rho_2[k]
    Lorentz_2=[]
    
    def lorentz(state, t):
        x, y, z = state 
        return sigma_2 * (y - x), x * (rho_2 - z) - y, x * y - beta_2 * z 
    
    for j in range(n):
      states = odeint(lorentz, estados_iniciales[j], t)
    
      Lorentz_2.append(states)
        
    Datos=np.concatenate((Lorentz_1,Lorentz_2))
    l=500
    diagramas_persistencia=[]
    imagenes_persistencia=[]
    
    for j in range(0,len(Datos)):
        diagramas_persistencia.append(ripser(Datos[j], maxdim=1)['dgms'])

    concatenacion_planos=[]
    for j in range(len(Datos)):
      if diagramas_persistencia[j][1].tolist()!=[]:
        concatenacion_planos.append(diagramas_persistencia[j][1].flatten())
      else:
        concatenacion_planos.append([])

    dimensiones_concatenacion=[]
    for j in range(len(Datos)):
      dimensiones_concatenacion.append(len(concatenacion_planos[j]))

    concatenacion_planos_padding=[]

    for j in range(len(Datos)):
      ceros=np.zeros(max(dimensiones_concatenacion)-len(concatenacion_planos[j]))
      concatenacion_planos_padding.append(np.concatenate((concatenacion_planos[j], ceros)))

    pca=PCA(n_components=2)

    X_scaled = StandardScaler().fit_transform(concatenacion_planos_padding)
    
    concatenacion_pca = pca.fit_transform(X_scaled)

    tipo_dato = [r'$\rho_1 = ' + str(rho_1) + r'$', r'$\rho_2 = ' + str(rho_2) + r'$']
    elem_tipo=10
    
    etiquetas=[]
    for i in range(len(tipo_dato)):
        etiquetas=etiquetas+list([tipo_dato[i]]*elem_tipo)
    
    colores = ['#603160', '#D49AD4', '#2AA58B', '#AEEEE2']
    
    plt.figure(figsize=(10, 7))
    
    for i in range(len(tipo_dato)):
        inicio, fin = i*elem_tipo, (i+1)*elem_tipo
        plt.scatter(concatenacion_pca[:,0][inicio:fin], concatenacion_pca[:,1][inicio:fin], c=colores[i], label=tipo_dato[i], s=100, alpha=0.95)
    

    plt.legend(fontsize=15)
    plt.title('PCA de la Vectorización. \n Contatenación: Comparación entre Atractores de Lorentz', fontsize=15)
    plt.xlabel(r"Componente Principal 1", fontsize=13)
    plt.ylabel(r"Componente Principal 2", fontsize=13)
    plt.grid(True, alpha=0.5)
    plt.xlim(-40,40)
    plt.ylim(-40,40)
    plt.savefig(f"PCA Comparación {k}.png", dpi=300, bbox_inches='tight')